# 260331 RAG 검색 평가 메트릭 (Retrieval Evaluation Metrics)

**4주차 Day 1** -- RAG 시스템에서 "검색"이 얼마나 잘 됐는지를 정량적으로 측정하는 방법을 배웁니다.

### 이번 수업 핵심 흐름
1. **골든 데이터셋** 만들기 -- 질문 + 정답 문서 + 정답 답변 (삼중 쌍)
2. **Hit Rate@K** -- K개 중 정답이 있는지 (있다/없다)
3. **MRR (Mean Reciprocal Rank)** -- 정답이 몇 등으로 나왔는지 (순위 반영)
4. **nDCG (normalized DCG)** -- 여러 정답 문서의 순위를 모두 반영
5. **Precision / Recall** -- (다음 시간에 이어서)

> 비유: 시험을 보려면 정답지가 필요하듯, 검색 성능을 측정하려면 "골든 데이터셋"이라는 정답지가 필요합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w4_rag_evaluation/llm_260331_rag_retrieval_basics.ipynb)

---
## 0. 환경 설정 (Colab Setup)

In [1]:
# --- Colab 사용 시 아래 주석 해제 ---
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# .env 파일에서 OPENAI_API_KEY 로드
# load_dotenv()

# 모델 설정 -- 임베딩과 LLM을 각각 정의
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

llm = ChatOpenAI(model=LLM_MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

---
## 1. 골든 데이터셋 준비

검색 성능을 측정하려면 **정답지**가 필요합니다.

- `documents`: 벡터 스토어에 들어갈 문서들 (도서관의 책)
- `qa_dataset`: 질문 + 정답 문서 ID + 정답 답변 (시험 문제와 정답지)

> 비유: 도서관에 책을 꽂아두고(documents), "이 질문을 하면 이 책을 찾아와야 해" 라는 정답지(qa_dataset)를 만드는 것과 같습니다.

In [4]:
# 문서 데이터: 벡터 스토어에 저장될 10개 문서
# 각 문서는 doc_id, title, content로 구성
documents = [
    {
        "doc_id": "doc_001",
        "title": "트랜스포머 아키텍처",
        "content": "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. "
                   "셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. "
                   "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."
    },
    {
        "doc_id": "doc_002",
        "title": "RAG 시스템 개요",
        "content": "RAG(Retrieval-Augmented Generation)는 검색과 생성을 결합한 기법입니다. "
                   "외부 지식 베이스에서 관련 문서를 검색한 후, 이를 컨텍스트로 활용하여 LLM이 답변을 생성합니다. "
                   "환각(hallucination)을 줄이고 최신 정보를 반영할 수 있는 장점이 있습니다."
    },
    {
        "doc_id": "doc_003",
        "title": "벡터 임베딩과 유사도 검색",
        "content": "벡터 임베딩은 텍스트를 고차원 벡터 공간에 매핑하는 기술입니다. "
                   "코사인 유사도를 사용하여 의미적으로 유사한 문서를 검색합니다. "
                   "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 검색에 활용됩니다."
    },
    {
        "doc_id": "doc_004",
        "title": "프롬프트 엔지니어링",
        "content": "프롬프트 엔지니어링은 LLM에게 효과적인 지시를 설계하는 기술입니다. "
                   "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있습니다. "
                   "시스템 프롬프트와 사용자 프롬프트를 구분하여 역할과 지시를 명확히 합니다."
    },
    {
        "doc_id": "doc_005",
        "title": "파인튜닝과 전이학습",
        "content": "파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. "
                   "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. "
                   "전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다."
    },
    {
        "doc_id": "doc_006",
        "title": "토큰화와 텍스트 전처리",
        "content": "토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. "
                   "BPE(Byte Pair Encoding), WordPiece, SentencePiece 등의 알고리즘이 있습니다. "
                   "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."
    },
    {
        "doc_id": "doc_007",
        "title": "LLM 평가 메트릭",
        "content": "LLM 평가에는 자동 메트릭과 인간 평가가 사용됩니다. "
                   "BLEU, ROUGE, BERTScore 등의 자동 메트릭은 참조 답변과의 유사도를 측정합니다. "
                   "LLM-as-Judge 방식은 다른 LLM을 활용하여 품질을 평가하는 최신 접근법입니다."
    },
    {
        "doc_id": "doc_008",
        "title": "청킹 전략",
        "content": "청킹은 긴 문서를 적절한 크기로 분할하는 전략입니다. "
                   "고정 크기 청킹, 의미 기반 청킹, 재귀적 청킹 등의 방법이 있습니다. "
                   "청크 크기와 오버랩은 검색 품질에 큰 영향을 미치며, 보통 500-1000 토큰을 사용합니다."
    },
    {
        "doc_id": "doc_009",
        "title": "하이브리드 검색",
        "content": "하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. "
                   "RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. "
                   "키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다."
    },
    {
        "doc_id": "doc_010",
        "title": "멀티모달 AI",
        "content": "멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. "
                   "GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. "
                   "CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."
    }
]

In [5]:
# 문서 목록 확인
for doc in documents:
    print(f"  {doc['doc_id']} : {doc['title']}")

  doc_001 : 트랜스포머 아키텍처
  doc_002 : RAG 시스템 개요
  doc_003 : 벡터 임베딩과 유사도 검색
  doc_004 : 프롬프트 엔지니어링
  doc_005 : 파인튜닝과 전이학습
  doc_006 : 토큰화와 텍스트 전처리
  doc_007 : LLM 평가 메트릭
  doc_008 : 청킹 전략
  doc_009 : 하이브리드 검색
  doc_010 : 멀티모달 AI


### QA 데이터셋 (골든 데이터셋)

각 항목은 **(질문, 정답 문서 ID, 정답 답변)**의 삼중 쌍입니다.

- `relevant_doc_ids`: 이 질문에 대해 검색되어야 하는 정답 문서
- `ground_truth`: 이 질문에 대한 정답 답변 (생성 평가 때 사용)

> 주의: q10처럼 정답 문서가 2개인 경우도 있습니다. 하나의 질문에 여러 관련 문서가 있을 수 있기 때문!

In [6]:
# QA 데이터셋: 질문 + 정답 문서 ID + 정답 답변
# relevant_doc_ids가 핵심 -- 검색이 이 문서를 찾아야 "정답"
qa_dataset = [
    {
        "query_id": "q01",
        "question": "트랜스포머의 핵심 메커니즘은 무엇인가요?",
        "relevant_doc_ids": ["doc_001"],
        "ground_truth": "트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다."
    },
    {
        "query_id": "q02",
        "question": "RAG 시스템의 장점은 무엇인가요?",
        "relevant_doc_ids": ["doc_002"],
        "ground_truth": "RAG는 환각을 줄이고 최신 정보를 반영할 수 있으며, 외부 지식 베이스를 활용하여 더 정확한 답변을 생성합니다."
    },
    {
        "query_id": "q03",
        "question": "벡터 유사도 검색에 사용되는 데이터베이스는?",
        "relevant_doc_ids": ["doc_003"],
        "ground_truth": "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 벡터 유사도 검색에 활용됩니다."
    },
    {
        "query_id": "q04",
        "question": "프롬프트 엔지니어링의 주요 기법은?",
        "relevant_doc_ids": ["doc_004"],
        "ground_truth": "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있으며, 시스템/사용자 프롬프트를 구분합니다."
    },
    {
        "query_id": "q05",
        "question": "효율적 파인튜닝 기법에는 어떤 것이 있나요?",
        "relevant_doc_ids": ["doc_005"],
        "ground_truth": "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다."
    },
    {
        "query_id": "q06",
        "question": "한국어 토큰화에 적합한 방법은?",
        "relevant_doc_ids": ["doc_006"],
        "ground_truth": "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."
    },
    {
        "query_id": "q07",
        "question": "LLM-as-Judge란 무엇인가요?",
        "relevant_doc_ids": ["doc_007"],
        "ground_truth": "LLM-as-Judge는 다른 LLM을 활용하여 생성 품질을 평가하는 최신 접근법입니다."
    },
    {
        "query_id": "q08",
        "question": "적절한 청크 크기는 얼마인가요?",
        "relevant_doc_ids": ["doc_008"],
        "ground_truth": "보통 500-1000 토큰 크기를 사용하며, 청크 크기와 오버랩이 검색 품질에 큰 영향을 미칩니다."
    },
    {
        "query_id": "q09",
        "question": "하이브리드 검색에서 결과를 병합하는 방법은?",
        "relevant_doc_ids": ["doc_009"],
        "ground_truth": "RRF(Reciprocal Rank Fusion)를 통해 키워드 검색과 벡터 검색 결과를 효과적으로 병합합니다."
    },
    {
        "query_id": "q10",
        "question": "검색과 생성을 결합하여 환각을 줄이는 기법은?",
        "relevant_doc_ids": ["doc_002", "doc_003"],
        "ground_truth": "RAG(Retrieval-Augmented Generation)는 검색으로 관련 문서를 찾고 LLM이 이를 기반으로 답변을 생성하여 환각을 줄입니다."
    },
    {
        "query_id": "q11",
        "question": "CLIP 모델의 활용 분야는?",
        "relevant_doc_ids": ["doc_010"],
        "ground_truth": "CLIP은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."
    },
    {
        "query_id": "q12",
        "question": "트랜스포머의 구조는 어떻게 되어있나요?",
        "relevant_doc_ids": ["doc_001"],
        "ground_truth": "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."
    }
]

In [7]:
# QA 데이터셋 미리보기: query_id, 질문, 정답 문서 ID
for qa in qa_dataset[:5]:
    print(f"  {qa['query_id']}: {qa['question'][:40]} ---> {qa['relevant_doc_ids']}")

  q01: 트랜스포머의 핵심 메커니즘은 무엇인가요? ---> ['doc_001']
  q02: RAG 시스템의 장점은 무엇인가요? ---> ['doc_002']
  q03: 벡터 유사도 검색에 사용되는 데이터베이스는? ---> ['doc_003']
  q04: 프롬프트 엔지니어링의 주요 기법은? ---> ['doc_004']
  q05: 효율적 파인튜닝 기법에는 어떤 것이 있나요? ---> ['doc_005']


---
## 2. 벡터 스토어 구축 & 검색 함수

문서를 FAISS 벡터 스토어에 넣고, 검색 함수를 만듭니다.

> 비유: 도서관에 책을 분류해서 꽂아넣는 과정입니다. 나중에 질문이 오면 가장 관련 있는 책을 꺼내옵니다.

In [8]:
# LangChain Document 객체로 변환 후 FAISS 벡터 스토어 생성
# FAISS = Facebook AI Similarity Search (벡터(숫자 배열)들 사이에서 가장 비슷한 것을 빠르게 찾아주는 검색 엔진)
# RAG에서의 역할: 질문 → 임베딩(벡터로 변환) → FAISS가 가장 유사한 문서 검색 → LLM에 전달 → 답변
# page_content = 문서 내용, metadata = 문서 ID와 제목
langchain_docs = [
    Document(
        page_content=doc['content'],
        metadata={"doc_id": doc['doc_id'], 'title': doc['title']}
    ) for doc in documents
]

# FAISS 벡터 스토어 생성 -- 문서를 임베딩하여 인덱싱
vectorstore = FAISS.from_documents(langchain_docs, embeddings)

# 인덱싱된 문서 수 확인 (10개여야 정상)
print(f"벡터 스토어에 인덱싱된 문서 수: {vectorstore.index.ntotal}")

벡터 스토어에 인덱싱된 문서 수: 10


In [9]:
def search_documents(query, k):
    """질의문(query)으로 벡터 스토어에서 k개 문서를 검색하는 함수

    similarity_search_with_score: 거리 기반 스코어 반환
    - 스코어가 낮을수록 = 거리가 가까울수록 = 더 관련 있는 문서
    (코사인 유사도와 반대! 주의)
    """
    results = vectorstore.similarity_search_with_score(query, k=k)
    retrieved = []
    for doc, score in results:
        retrieved.append({
            'doc_id': doc.metadata['doc_id'],
            'title': doc.metadata['title'],
            'content': doc.page_content,
            'score': float(score)  # numpy -> float 변환
        })
    return retrieved

### 전체 QA에 대해 검색 결과 캐싱

매번 검색하면 API 비용이 드니까, 한 번에 모든 질문의 검색 결과를 저장해둡니다.

In [10]:
# 모든 QA 질문에 대해 검색 결과를 미리 캐싱 (k=5)
# search_results_cache[query_id] = [검색된 문서 리스트]
search_results_cache = {}
for qa in qa_dataset:
    results = search_documents(qa['question'], k=5)
    search_results_cache[qa['query_id']] = results

print(f"캐싱 완료: {len(search_results_cache)}개 쿼리")

캐싱 완료: 12개 쿼리


### 검색 결과 살펴보기

q04(프롬프트 엔지니어링 질문)를 예시로 검색 결과를 확인합니다.

In [11]:
# 샘플로 q04의 검색 결과 확인
sample_q = qa_dataset[3]  # q04: 프롬프트 엔지니어링의 주요 기법은?
print(f"질문: {sample_q['question']}")
print(f"정답 문서: {sample_q['relevant_doc_ids']}")
print("\n--- 검색 결과 (거리 순, 낮을수록 관련도 높음) ---")
for i, r in enumerate(search_results_cache[sample_q['query_id']]):
    print(f"  {i+1}위: {r['doc_id']} ({r['title']}) | score={r['score']:.4f}")

질문: 프롬프트 엔지니어링의 주요 기법은?
정답 문서: ['doc_004']

--- 검색 결과 (거리 순, 낮을수록 관련도 높음) ---
  1위: doc_004 (프롬프트 엔지니어링) | score=0.6668
  2위: doc_001 (트랜스포머 아키텍처) | score=1.4116
  3위: doc_005 (파인튜닝과 전이학습) | score=1.4469
  4위: doc_002 (RAG 시스템 개요) | score=1.4781
  5위: doc_010 (멀티모달 AI) | score=1.5166


---
## 3. 데이터셋 확장 & 검증

새 문서와 QA 쌍을 추가한 뒤, 데이터셋의 무결성을 검증합니다.

> 비유: 도서관에 새 책을 추가하고, 정답지에 적힌 책 번호가 실제로 도서관에 있는지 확인하는 과정입니다.

In [12]:
# 새 문서 2개 추가
new_documents = [
    {
        "doc_id": "doc_011",
        "title": "어텐션 메커니즘의 변형",
        "content": "어텐션 메커니즘에는 다양한 변형이 있습니다. "
                   "Flash Attention은 메모리 효율적인 어텐션 계산을 제공하며, "
                   "Sparse Attention은 긴 시퀀스 처리에 효과적입니다."
    },
    {
        "doc_id": "doc_012",
        "title": "LLM 양자화 기법",
        "content": "양자화는 모델의 가중치를 낮은 비트로 표현하여 메모리와 연산을 줄이는 기법입니다. "
                   "GPTQ, AWQ, GGUF 등의 포맷이 있으며, 4비트 양자화가 널리 사용됩니다."
    }
]

# 새 QA 쌍 2개 추가
new_qa_pairs = [
    {
        "query_id": "q13",
        "question": "메모리 효율적인 어텐션 방법은?",
        "relevant_doc_ids": ["doc_011", "doc_001"],
        "ground_truth": "Flash Attention은 메모리 효율적인 어텐션 계산을 제공합니다."
    },
    {
        "query_id": "q14",
        "question": "LLM 모델 크기를 줄이는 양자화 포맷은?",
        "relevant_doc_ids": ["doc_012"],
        "ground_truth": "GPTQ, AWQ, GGUF 등의 양자화 포맷이 있으며 4비트 양자화가 널리 사용됩니다."
    }
]

In [13]:
# 기존 + 새 데이터 합치기
all_docs = documents + new_documents
all_qa = qa_dataset + new_qa_pairs

# 벡터 스토어도 새 문서 포함하여 재구축해야 함!
langchain_docs = [
    Document(
        page_content=doc['content'],
        metadata={"doc_id": doc['doc_id'], 'title': doc['title']}
    ) for doc in all_docs
]
vectorstore = FAISS.from_documents(langchain_docs, embeddings)
print(f"벡터 스토어 문서 수: {vectorstore.index.ntotal}")

# 검색 결과 캐시도 전체 QA로 업데이트
search_results_cache = {}
for qa in all_qa:
    results = search_documents(qa['question'], k=5)
    search_results_cache[qa['query_id']] = results

print(f"캐싱 완료: {len(search_results_cache)}개 쿼리")

벡터 스토어 문서 수: 12
캐싱 완료: 14개 쿼리


In [14]:
def validate_dataset(docs, qa_pairs):
    """QA 데이터셋의 relevant_doc_ids가 실제 문서에 존재하는지 검증

    비유: 시험 정답지에 적힌 교과서 페이지가 실제로 존재하는지 확인
    """
    doc_ids = {d['doc_id'] for d in docs}  # 전체 문서 ID 집합
    errors = []
    for qa in qa_pairs:
        for rid in qa['relevant_doc_ids']:
            if rid not in doc_ids:
                errors.append(f"{qa['query_id']} : {rid} 문서 없음")

    if errors:
        print(f"오류 {len(errors)}건: {errors}")
    else:
        print("통과: 모든 relevant_doc_ids가 문서에 존재합니다.")
    return len(errors)

# 검증 실행
validate_dataset(all_docs, all_qa)

통과: 모든 relevant_doc_ids가 문서에 존재합니다.


0

---
## 4. Hit Rate@K -- 가장 단순한 검색 평가

**아이디어**: K개 검색 결과 중에 정답 문서가 **하나라도 있으면 1, 없으면 0**

> 비유: 시험에서 5지선다 문제를 풀었을 때, 정답이 보기에 있기만 하면 OK

**장점**: 매우 직관적이고 계산이 간단  
**한계**: 순위를 무시함 -- 1등으로 찾든 5등으로 찾든 같은 점수  
**한계**: 정답이 여러 개일 때 하나만 맞아도 1점

In [15]:
def hit_rate_at_k(query_id, k):
    """개별 쿼리의 Hit Rate@K 계산

    K개 검색 결과 중 정답 문서가 하나라도 있으면 1, 없으면 0

    next() 사용 이유: 리스트 컴프리헨션은 전체를 순회하지만
    next()는 첫 번째 매칭에서 멈춤 -- 데이터가 많을 때 효율적!
    """
    # next()로 해당 query_id의 QA를 빠르게 찾기
    qa = next(q for q in qa_dataset if q['query_id'] == query_id)

    # 정답 문서 ID 집합 (set으로 중복 제거)
    relevant_ids = set(qa['relevant_doc_ids'])

    # 검색 결과에서 K개만 가져오기
    retrieved = search_results_cache[query_id][:k]
    retrieved_ids = {r['doc_id'] for r in retrieved}

    # 교집합(&)이 있으면 1, 없으면 0
    return 1 if relevant_ids & retrieved_ids else 0

In [16]:
def average_hit_rate_at_k(k):
    """전체 QA 데이터셋의 평균 Hit Rate@K

    개별 쿼리의 1/0 값을 모두 더한 후 쿼리 수로 나눔
    """
    hits = [hit_rate_at_k(qa['query_id'], k) for qa in qa_dataset]
    return sum(hits) / len(hits)

In [17]:
# 각 쿼리별 Hit Rate@3 확인 -- 정답과 검색 결과를 나란히 출력
for qa in qa_dataset:
    hit = hit_rate_at_k(qa['query_id'], k=3)
    retrieved_ids = [r['doc_id'] for r in search_results_cache[qa['query_id']][:3]]
    status = 'HIT' if hit else 'MISS'
    print(f"[{status}] {qa['query_id']}: 정답 = {qa['relevant_doc_ids']} | 검색 = {retrieved_ids}")

[HIT] q01: 정답 = ['doc_001'] | 검색 = ['doc_001', 'doc_010', 'doc_005']
[HIT] q02: 정답 = ['doc_002'] | 검색 = ['doc_002', 'doc_004', 'doc_009']
[HIT] q03: 정답 = ['doc_003'] | 검색 = ['doc_003', 'doc_009', 'doc_012']
[HIT] q04: 정답 = ['doc_004'] | 검색 = ['doc_004', 'doc_001', 'doc_005']
[HIT] q05: 정답 = ['doc_005'] | 검색 = ['doc_005', 'doc_003', 'doc_011']
[HIT] q06: 정답 = ['doc_006'] | 검색 = ['doc_006', 'doc_009', 'doc_008']
[HIT] q07: 정답 = ['doc_007'] | 검색 = ['doc_007', 'doc_004', 'doc_002']
[HIT] q08: 정답 = ['doc_008'] | 검색 = ['doc_008', 'doc_003', 'doc_001']
[HIT] q09: 정답 = ['doc_009'] | 검색 = ['doc_009', 'doc_002', 'doc_008']
[HIT] q10: 정답 = ['doc_002', 'doc_003'] | 검색 = ['doc_002', 'doc_009', 'doc_008']
[HIT] q11: 정답 = ['doc_010'] | 검색 = ['doc_010', 'doc_005', 'doc_006']
[HIT] q12: 정답 = ['doc_001'] | 검색 = ['doc_001', 'doc_010', 'doc_004']


In [18]:
# K값에 따른 평균 Hit Rate 변화 확인
# K가 클수록 정답이 포함될 확률이 높아짐
for k in [1, 3, 5]:
    score = average_hit_rate_at_k(k)
    print(f"Average Hit Rate@{k} : {score:.4f}")

Average Hit Rate@1 : 1.0000
Average Hit Rate@3 : 1.0000
Average Hit Rate@5 : 1.0000


### Hit Rate의 한계 확인

새로운 (어려운) 쿼리로 테스트하면 K에 따라 차이가 발생합니다.

In [19]:
# 새로운 테스트용 QA -- 기존보다 간접적인 질문들
new_qa = [
    {"query_id": "q13", "question": "Flash Attention의 장점은?",
     "relevant_doc_ids": ["doc_001"],
     "ground_truth": "Flash Attention은 메모리 효율적인 어텐션 계산을 제공합니다."},
    {"query_id": "q14", "question": "양자화로 모델 크기를 줄이는 방법은?",
     "relevant_doc_ids": ["doc_005"],
     "ground_truth": "LoRA, QLoRA 등의 효율적 파인튜닝과 양자화 기법으로 모델 크기를 줄입니다."},
    {"query_id": "q15", "question": "RAG에서 청크 크기가 검색에 미치는 영향은?",
     "relevant_doc_ids": ["doc_008", "doc_002"],
     "ground_truth": "청크 크기는 검색 품질에 큰 영향을 미치며 보통 500-1000 토큰을 사용합니다."},
]

# 새 쿼리의 검색 결과도 캐싱
for qa in new_qa:
    search_results_cache[qa['query_id']] = search_documents(qa['question'], k=5)

In [20]:
# new_qa 기반으로 hit_rate 재정의 (new_qa 데이터셋 사용)
def hit_rate_at_k_new(query_id, k):
    qa = next(q for q in new_qa if q['query_id'] == query_id)
    relevant_ids = set(qa['relevant_doc_ids'])
    retrieved = search_results_cache[query_id][:k]
    retrieved_ids = {r['doc_id'] for r in retrieved}
    return 1 if relevant_ids & retrieved_ids else 0

def average_hit_rate_at_k_new(k):
    hits = [hit_rate_at_k_new(qa['query_id'], k) for qa in new_qa]
    return sum(hits) / len(hits)

# K=1, 3, 5로 비교 -- K가 작을수록 성능 차이가 드러남
for qa in new_qa:
    for k in [1, 3, 5]:
        hit = hit_rate_at_k_new(qa['query_id'], k)
        status = 'hit' if hit else 'miss'
        print(f"  {qa['query_id']} (@K={k}): {status} | 정답: {qa['relevant_doc_ids']}")
    print(f"  HR@1={average_hit_rate_at_k_new(1):.2f}, HR@3={average_hit_rate_at_k_new(3):.2f}, HR@5={average_hit_rate_at_k_new(5):.2f}")
    print()

  q13 (@K=1): miss | 정답: ['doc_001']
  q13 (@K=3): miss | 정답: ['doc_001']
  q13 (@K=5): hit | 정답: ['doc_001']
  HR@1=0.33, HR@3=0.67, HR@5=1.00

  q14 (@K=1): miss | 정답: ['doc_005']
  q14 (@K=3): hit | 정답: ['doc_005']
  q14 (@K=5): hit | 정답: ['doc_005']
  HR@1=0.33, HR@3=0.67, HR@5=1.00

  q15 (@K=1): hit | 정답: ['doc_008', 'doc_002']
  q15 (@K=3): hit | 정답: ['doc_008', 'doc_002']
  q15 (@K=5): hit | 정답: ['doc_008', 'doc_002']
  HR@1=0.33, HR@3=0.67, HR@5=1.00



---
## 5. MRR (Mean Reciprocal Rank) -- 순위를 반영하는 평가

**아이디어**: 정답 문서가 **몇 등으로** 나왔는지를 반영

- 1등: 1/1 = 1.0
- 2등: 1/2 = 0.5
- 3등: 1/3 = 0.33
- K 안에 없음: 0

> 비유: 구글 검색에서 첫 번째 결과가 정답이면 좋고, 10페이지에 있으면 별로인 것과 같습니다.

**기준**: MRR >= 0.5이면 양호, >= 0.8이면 우수  
**Hit Rate와 차이**: Hit Rate는 있다/없다만 보지만, MRR은 순위도 반영

In [21]:
def reciprocal_rank(query_id, k):
    """개별 쿼리의 Reciprocal Rank 계산

    첫 번째로 등장하는 정답 문서의 순위의 역수를 반환
    예: 1등 -> 1.0, 2등 -> 0.5, 3등 -> 0.33
    """
    qa = next(q for q in qa_dataset if q['query_id'] == query_id)
    relevant_ids = set(qa['relevant_doc_ids'])
    retrieved = search_results_cache[query_id][:k]

    # enumerate(_, 1): 1부터 시작하는 순위
    for rank, result in enumerate(retrieved, 1):
        if result['doc_id'] in relevant_ids:
            return 1.0 / rank  # 첫 번째 정답 문서의 역수 반환

    return 0.0  # K 안에 정답 없음


def mean_reciprocal_rank(k):
    """전체 QA 데이터셋의 평균 MRR"""
    rr_scores = [reciprocal_rank(qa['query_id'], k) for qa in qa_dataset]
    return sum(rr_scores) / len(rr_scores)

In [22]:
# MRR@5 계산 -- 1.0이면 모든 쿼리에서 정답이 1등으로 검색됨
print(f"MRR@5 = {mean_reciprocal_rank(k=5):.4f}")

MRR@5 = 1.0000


In [23]:
# 쿼리별 RR 상세 분석 -- DataFrame으로 깔끔하게 출력
rows = []
for qa in qa_dataset:
    rr = reciprocal_rank(qa['query_id'], 5)
    rows.append({
        'query_id': qa['query_id'],
        'question': qa['question'],
        'RR@5': round(rr, 4)
    })

df_rr = pd.DataFrame(rows)
df_rr

,query_id,question,RR@5
0,q01,트랜스포머의 핵심 메커니즘은 무엇인가요?,1.0
1,q02,RAG 시스템의 장점은 무엇인가요?,1.0
2,q03,벡터 유사도 검색에 사용되는 데이터베이스는?,1.0
3,q04,프롬프트 엔지니어링의 주요 기법은?,1.0
4,q05,효율적 파인튜닝 기법에는 어떤 것이 있나요?,1.0
5,q06,한국어 토큰화에 적합한 방법은?,1.0
6,q07,LLM-as-Judge란 무엇인가요?,1.0
7,q08,적절한 청크 크기는 얼마인가요?,1.0
8,q09,하이브리드 검색에서 결과를 병합하는 방법은?,1.0
9,q10,검색과 생성을 결합하여 환각을 줄이는 기법은?,1.0


---
## 6. nDCG (normalized Discounted Cumulative Gain) -- 여러 정답의 순위를 모두 반영

**MRR의 한계**: 첫 번째 정답 문서만 봄. 정답이 2개인 q10 같은 경우를 제대로 평가 못함.

**nDCG 아이디어**: 모든 정답 문서의 순위를 **등급별 가중치**로 반영

### DCG 공식
```
DCG = sum( relevance_score / log2(rank + 1) )
```
- 높은 순위(1등)일수록 log 값이 작아서 점수가 높아짐
- relevance_score: 관련 있으면 1, 없으면 0 (혹은 0~5 등급)

### nDCG = DCG / IDCG
- IDCG = **이상적인** DCG (정답이 완벽한 순서로 나왔을 때)
- nDCG는 항상 0~1 사이 값

> 비유: 키(cm), 몸무게(kg), 신발(mm) 처럼 단위가 다른 값을 0~1 사이로 **정규화(normalize)**하는 것과 같습니다.  
> 정답 문서가 3개든 5개든, nDCG로 같은 스케일에서 비교할 수 있습니다.

### MRR vs nDCG
| | MRR | nDCG |
|---|---|---|
| 보는 문서 수 | 첫 번째 정답만 | 모든 정답 문서 |
| 관련성 | 있다/없다 (binary) | 등급별 점수 가능 |
| 적합한 경우 | FAQ, QA | 검색 엔진, 추천 시스템 |

- 둘 다 "정답이 몇 등인가"를 보는 건 같은데, 정답이 1개냐 여러 개냐에서 구분
- **MRR (Mean Reciprocal Rank):**
첫 번째 정답만 본다. "정답을 처음 찾기까지 얼마나 걸렸어?"
- 검색 결과: [X, O, X, O, X]  ← 2번째에서 처음 발견 (MRR = 1/2 = 0.5)
- 나머지 정답(4번째 O)은 무시. 빠르게 하나만 찾으면 되는 FAQ 검색에 적합
- **nDCG (normalized DCG)** 모든 정답의 위치를 본다. "정답들이 전체적으로 얼마나 위에 몰려있어?"
- 검색 결과: [X, O, X, O, X]  ← 2번째, 4번째 둘 다 계산 (각 위치마다 log로 가중치 부여 → 위에 있을수록 점수 높음)

In [24]:
def dcg_at_k(relevances, k):
    """DCG (Discounted Cumulative Gain) 계산

    각 문서의 관련성 점수를 순위에 따라 할인(discount)하여 합산
    - 순위가 높을수록(1등에 가까울수록) 높은 가중치
    - log2(i+2)로 나누는 이유: log2(1)=0이면 0으로 나누기 에러!
      i=0일 때 log2(2)=1, i=1일 때 log2(3)=1.58, ...
    """
    relevances = relevances[:k]
    dcg = 0.0
    for i, rel in enumerate(relevances):
        dcg += rel / np.log2(i + 2)  # i+2: 0번째 -> log2(2)=1
    return dcg

In [25]:
def ndcg_at_k(query_id, k):
    """nDCG@K 계산: DCG를 이상적 DCG(IDCG)로 정규화

    1) 검색 결과에서 관련성 리스트 생성 (정답이면 1, 아니면 0)
    2) DCG 계산
    3) IDCG 계산 (정답이 완벽히 앞에 나왔을 때의 DCG)
    4) nDCG = DCG / IDCG
    """
    qa = next(q for q in qa_dataset if q['query_id'] == query_id)
    relevant_ids = set(qa['relevant_doc_ids'])
    retrieved = search_results_cache[query_id][:k]

    # 실제 검색 결과의 관련성 리스트: [1, 0, 0, 1, 0] 등
    relevances = [1 if r['doc_id'] in relevant_ids else 0 for r in retrieved]
    dcg = dcg_at_k(relevances, k)

    # 이상적 관련성: 정답 문서가 모두 앞에 나오는 경우
    # 예: 정답 2개, k=5 -> [1, 1, 0, 0, 0]
    total_relevant = len(relevant_ids)
    ideal_relevances = [1] * min(total_relevant, k) + [0] * max(0, k - total_relevant)
    idcg = dcg_at_k(ideal_relevances, k)

    if idcg == 0:
        return 0.0
    return dcg / idcg

In [26]:
def average_ndcg_at_k(k):
    """전체 QA 데이터셋의 평균 nDCG@K"""
    scores = [ndcg_at_k(qa['query_id'], k) for qa in qa_dataset]
    return sum(scores) / len(scores)

In [27]:
# K=1, 3, 5에 따른 nDCG 비교
# K=1이면 MRR과 동일한 결과, K가 커질수록 여러 정답 문서 반영
for k in [1, 3, 5]:
    score = average_ndcg_at_k(k)
    print(f"nDCG@{k} : {score:.4f}")

nDCG@1 : 1.0000
nDCG@3 : 0.9678
nDCG@5 : 0.9898


In [28]:
# 쿼리별 nDCG@5 상세 -- 정답 수가 2개인 q10에 주목!
# q10만 1.0이 아닌 이유: 정답 2개(doc_002, doc_003) 중
# 하나가 상위권에 없거나 순서가 맞지 않아서
for qa in qa_dataset:
    score = ndcg_at_k(qa['query_id'], 5)
    print(f"{qa['query_id']} : nDCG@5 = {score:.4f} | 정답 수 = {len(qa['relevant_doc_ids'])}")

q01 : nDCG@5 = 1.0000 | 정답 수 = 1
q02 : nDCG@5 = 1.0000 | 정답 수 = 1
q03 : nDCG@5 = 1.0000 | 정답 수 = 1
q04 : nDCG@5 = 1.0000 | 정답 수 = 1
q05 : nDCG@5 = 1.0000 | 정답 수 = 1
q06 : nDCG@5 = 1.0000 | 정답 수 = 1
q07 : nDCG@5 = 1.0000 | 정답 수 = 1
q08 : nDCG@5 = 1.0000 | 정답 수 = 1
q09 : nDCG@5 = 1.0000 | 정답 수 = 1
q10 : nDCG@5 = 0.8772 | 정답 수 = 2
q11 : nDCG@5 = 1.0000 | 정답 수 = 1
q12 : nDCG@5 = 1.0000 | 정답 수 = 1


---
## 7. Precision / Recall (예고)

다음 시간(260401)에 이어서 배울 내용입니다.

> 비유: 코로나 검사에서
> - **Precision** (정밀도): 양성 판정 중 실제 양성 비율 -- "찾은 것 중 진짜 정답은 몇 개?"
> - **Recall** (재현율): 실제 양성 중 양성 판정 비율 -- "전체 정답 중 몇 개를 찾았나?"

---
## 정리: 검색 평가 메트릭 비교

| 메트릭 | 핵심 질문 | 순위 반영 | 다중 정답 반영 | 계산 복잡도 |
|---|---|---|---|---|
| **Hit Rate@K** | K개 중 정답 있나? | X | X (하나만 있으면 OK) | 매우 간단 |
| **MRR** | 정답이 몇 등? | O (첫 번째 정답) | X (첫 번째만) | 간단 |
| **nDCG** | 모든 정답이 몇 등? | O (모든 정답) | O | 보통 |
| **Precision/Recall** | 찾은 것 중 진짜는? / 진짜 중 찾은 것은? | - | O | 간단 |

**실무 팁** (강사 코멘트):
- 데이터가 많으면 샘플링해서 평가해도 됨
- 골든 데이터셋은 보통 100개 이상 준비
- 골든 데이터셋의 relevance score는 LLM으로 자동 생성하거나, 클릭 로그로 대체하기도 함